# 2.2 Refactorización de P2

Lo elegido en `1.2-dm-p2-exploracion-riesgo` pasa a funciones y a una clase de experimento. Al final se compara con `mibici_equipo39/modeling/p2.py`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from mibici_equipo39 import config as C
from mibici_equipo39.features import FEATURE_COLUMNS, load_p2_targets, load_tables

metadata, X, _ = load_tables()
targets = load_p2_targets()

## Modelos

Las dos fábricas devuelven un estimador de scikit-learn con `fit` / `predict_proba`, así el experimento trata igual al baseline y al candidato. Los hiperparámetros salen de `config.py`.

In [2]:
NUMERIC, CATEGORICAL = list(FEATURE_COLUMNS), ["station_id"]

def make_baseline():
    return DummyClassifier(strategy="prior")

def make_candidate():
    columns = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL),
    ])
    return make_pipeline(columns, LogisticRegression(**C.P2_LOGISTIC_PARAMS))

> **Por qué un `Pipeline`.** El imputador, el escalador y el one-hot quedan dentro del modelo, así que se ajustan solo con train al llamar `fit` y nunca ven validación ni prueba.

## Umbral y métricas

In [3]:
def counts(y, alert):
    tn, fp, fn, tp = confusion_matrix(y, alert, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1

def choose_threshold(y, score, grid=C.P2_THRESHOLDS, min_recall=C.P2_MIN_RECALL):
    table = pd.DataFrame([(t, *counts(y, score >= t)) for t in grid], columns=["threshold", "precision", "recall", "f1"])
    ok = table.loc[table["recall"].ge(min_recall)]
    pool, column, rule = (ok, "precision", "recall_floor") if len(ok) else (table, "f1", "max_f1")
    return float(pool.loc[pool[column].eq(pool[column].max()), "threshold"].max()), rule

def classification_metrics(y, score, threshold):
    precision, recall, f1 = counts(y, score >= threshold)
    values = {"recall": recall, "precision": precision, "f1": f1, "support_positive": int(np.sum(y))}
    if values["support_positive"]:
        values["ap"] = average_precision_score(y, score)
    return values

> **Casos borde.** Si ningún umbral llega a 80% de recall se usa mayor F1 en lugar de fallar, y si un conjunto no tiene positivos AP se omite en vez de inventar un valor.

## Experimento

In [4]:
frame = pd.concat([metadata, X, targets], axis=1)
frame = frame.loc[frame["capacity"].notna()].reset_index(drop=True)
train = frame.loc[frame["split"].eq("train")]
val = frame["split"].eq("validation").to_numpy()

notebook_scores, rows = {}, {}
for kind, factory in [("baseline", make_baseline), ("candidato", make_candidate)]:
    row = {}
    for risk, target in [("bikes", "y_bikes"), ("docks", "y_docks")]:
        y = frame[target].astype(int).to_numpy()
        score = factory().fit(train[NUMERIC + CATEGORICAL], train[target].astype(int)).predict_proba(frame[NUMERIC + CATEGORICAL])[:, 1]
        notebook_scores[(kind, risk)] = score
        threshold, rule = choose_threshold(y[val], score[val])
        row[f"threshold_{risk}"] = threshold
        row.update({f"val_{k}_{risk}": v for k, v in classification_metrics(y[val], score[val], threshold).items()})
    row["val_macro_f1"] = (row["val_f1_bikes"] + row["val_f1_docks"]) / 2
    rows[kind] = row
pd.DataFrame(rows).T[["threshold_bikes", "val_recall_bikes", "val_precision_bikes", "threshold_docks", "val_recall_docks", "val_precision_docks", "val_macro_f1"]].round(4)

,threshold_bikes,val_recall_bikes,val_precision_bikes,threshold_docks,val_recall_docks,val_precision_docks,val_macro_f1
baseline,0.95,0.000,0.0000,0.95,0.0000,0.0000,0.0000
candidato,0.65,0.818,0.0227,0.65,0.8122,0.0303,0.0513


> **Mismos números que la exploración.** Prueba se calcula solo dentro del run de MLflow.

## Paso al módulo

| Pieza | Destino |
|---|---|
| `proxy_labels`, `make_capacity_proxy_targets` | `mibici_equipo39/features.py` |
| `make_baseline`, `make_candidate`, `choose_threshold`, `classification_metrics`, `P2Experiment` | `mibici_equipo39/modeling/p2.py` |
| Rejilla, recall mínimo, ocupación, margen e hiperparámetros | `config.py` |
| Curva precisión-recall | `mibici_equipo39/plots.py` |

Se ejecuta con `python -m mibici_equipo39.modeling.p2 --executor personaN` o con `mlflow run . -e p2`.

In [5]:
from mibici_equipo39.modeling.p2 import P2Experiment

module = P2Experiment(metadata, X, targets).fit()
for kind in ["baseline", "candidato"]:
    for risk in ["bikes", "docks"]:
        assert np.allclose(notebook_scores[(kind, risk)], module.scores[kind][risk])
print("notebook y módulo dan los mismos puntajes; umbrales:", module.thresholds, "; seleccionado:", module.selected)

notebook y módulo dan los mismos puntajes; umbrales: {'baseline': {'bikes': {'threshold': 0.95, 'rule': 'max_f1'}, 'docks': {'threshold': 0.95, 'rule': 'max_f1'}}, 'candidato': {'bikes': {'threshold': 0.65, 'rule': 'recall_floor'}, 'docks': {'threshold': 0.65, 'rule': 'recall_floor'}}} ; seleccionado: candidato
